# Task B — SOLUTION NOTEBOOK
> **BAFAD Accelerated Research Track · Fall 2026 · INSTRUCTOR USE ONLY**

Complete Python translations of STAT 270 Assignment 1, Questions 2, 3, and 4.
Expected numerical results are noted in comments. Do not distribute to students.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde

NAVY = '#021B3A'
BLUE = '#0EA5E9'
plt.rcParams.update({'figure.dpi': 100, 'axes.spines.top': False, 'axes.spines.right': False})

flights  = pd.read_csv('../data/flights.csv')
airports = pd.read_csv('../data/airports.csv')

print(f'flights:  {flights.shape[0]:,} rows × {flights.shape[1]} columns')
print(f'airports: {airports.shape[0]} rows × {airports.shape[1]} columns')
flights.head(3)

---
## Question 2 — Frequency Tables and Proportions (10 pts)

### Part A (4 pts) — Flights per carrier

In [ ]:
# R: flights |> count(carrier, sort=TRUE) |> mutate(prop = n / sum(n))
carrier_counts = (
    flights['carrier']
    .value_counts()
    .reset_index()
)
carrier_counts.columns = ['carrier', 'n']
carrier_counts['prop'] = carrier_counts['n'] / carrier_counts['n'].sum()

carrier_counts
# Expected: F9 leads with 74 flights (7.4%), followed by B6 (72, 7.2%), US (71, 7.1%)

**Sample answer:** F9 operated the most flights (74), representing approximately 7.4% of all flights in the dataset. The carrier distribution is fairly even across the 16 carriers, with no single carrier dominating more than about 7–8% of total flights.

**Grading note (4 pts):** 2 pts for correct counts in descending order; 1 pt for a `prop` column that sums to 1.0; 1 pt for written answer naming the top carrier. Accept any equivalent computation (`groupby`, `value_counts`, manual dict). Deduct 1 pt if result is not sorted.

### Part B (6 pts) — Late arrival rate per carrier

In [ ]:
# R: flights |> group_by(carrier) |>
#      summarise(late_rate = mean(arr_delay > 0, na.rm=TRUE)) |>
#      arrange(desc(late_rate))
late_arrivals = (
    flights
    .groupby('carrier')['arr_delay']
    .apply(lambda x: (x > 0).mean())
    .reset_index()
    .rename(columns={'arr_delay': 'late_rate'})
    .sort_values('late_rate', ascending=False)
    .reset_index(drop=True)
)

late_arrivals
# Expected: AS tops the list (~73.6%), followed by UA (~70.6%) and YV (~70.5%)

**Sample answer:** AS (Alaska Airlines) had the worst on-time arrival record in this sample, with approximately 73.6% of its flights arriving late. This is notable because AS is generally regarded as a top-performing carrier nationally — the result is likely driven by the small sample size (the synthetic dataset uses only 1,000 flights).

**Grading note (6 pts):** 3 pts for correct `late_rate` values (within rounding); 2 pts for correct descending sort; 1 pt for written answer naming the worst-performing carrier. Accept `na.rm` equivalent (`skipna=True` is default in pandas, so `(x > 0).mean()` handles NaN correctly already — no deduction if students omit explicit NA handling).

---
## Question 3 — Histogram, Density, and Shape (15 pts)

### Part A (7 pts) — Density histogram + KDE

In [ ]:
df_filtered = flights[
    (flights['dep_delay'] >= -60) & (flights['dep_delay'] <= 180)
].dropna(subset=['dep_delay']).copy()

fig, ax = plt.subplots(figsize=(9, 5))

# Density-normalised histogram (matches R's after_stat(density))
ax.hist(df_filtered['dep_delay'],
        bins=40,
        density=True,
        color=NAVY,
        alpha=0.85,
        edgecolor='white',
        linewidth=0.5,
        label='Histogram')

# KDE overlay (matches R's geom_density)
x_range = np.linspace(df_filtered['dep_delay'].min(), df_filtered['dep_delay'].max(), 300)
kde = gaussian_kde(df_filtered['dep_delay'])
ax.plot(x_range, kde(x_range), color=BLUE, linewidth=2, label='KDE')

ax.set_xlabel('Departure Delay (minutes)')
ax.set_ylabel('Density')
ax.set_title('Distribution of Departure Delays (−60 to 180 min)')
ax.legend()
plt.tight_layout()
plt.show()

**Grading note (7 pts):**
- 2 pts: correct filter (`>= -60` and `<= 180`)
- 2 pts: `density=True` on histogram (or equivalent manual normalisation)
- 1 pt: NAVY histogram colour
- 1 pt: BLUE KDE line (accept `scipy.stats.gaussian_kde` or `pandas .plot.kde()`)
- 1 pt: axis labels and title present

Accept alternative KDE methods: `df_filtered['dep_delay'].plot.kde(ax=ax, color=BLUE)` is equivalent.

### Part B (8 pts) — Descriptive statistics and written interpretation

In [ ]:
median_delay = df_filtered['dep_delay'].median()
iqr_delay    = df_filtered['dep_delay'].quantile(0.75) - df_filtered['dep_delay'].quantile(0.25)
print(f'Median departure delay: {median_delay:.1f} min')
print(f'IQR of departure delay: {iqr_delay:.1f} min')
# Expected: Median = 11.0 min, IQR = 53.8 min

**Sample written interpretation (full marks):**

> The distribution of departure delays is unimodal and strongly right-skewed. The centre, measured by the median, is approximately 11 minutes — meaning half of all flights depart at least 11 minutes late. The spread is wide: the IQR is roughly 54 minutes, indicating substantial variability. The long right tail reflects the relatively rare but severe delays (up to 180 minutes) that pull the mean above the median. The histogram and KDE confirm a large mass of flights clustered around small positive delays, with a thin tail extending toward very long delays.

**Grading note (8 pts):** 3 pts for correct median and IQR values (within 0.5 min); 5 pts for written answer addressing all four: shape (right-skewed), modality (unimodal), centre (median ~11 min), and spread (IQR ~54 min). Deduct 1 pt each for missing dimension.

---
## Question 4 — Grouped Summaries by Origin Airport (15 pts)

### Part A (7 pts) — Summary table

In [ ]:
from scipy.stats import iqr as scipy_iqr

# R: flights |> group_by(origin) |> summarise(mean, median, sd, IQR of arr_delay)
origin_summary = (
    flights
    .groupby('origin')['arr_delay']
    .agg(
        mean_arr   = 'mean',
        median_arr = 'median',
        sd_arr     = 'std',
        iqr_arr    = lambda x: x.quantile(0.75) - x.quantile(0.25)
    )
    .round(2)
)

origin_summary
# Expected:
#         mean_arr  median_arr  sd_arr  iqr_arr
# EWR       16.34        15.0   43.27     56.0
# JFK       20.06        19.0   43.27     58.0
# LGA       15.10        15.0   42.42     57.5

**Grading note (7 pts):** 1 pt per correct column (4 columns × 1 pt) + 2 pts for correct groupby structure (deduct 1 pt if result not grouped by `origin`; deduct 1 pt if NAs not handled — pandas `.mean()` skips NaN by default so this is usually fine). Accept `scipy.stats.iqr` as alternative for IQR.

### Part B (8 pts) — Written interpretation

**Sample answer (full marks):**

> JFK has the highest mean arrival delay (≈20.1 min), while LGA has the lowest (≈15.1 min). EWR falls in the middle at ≈16.3 min.
>
> For all three airports, the mean is higher than the median (EWR: 16.3 vs 15.0; JFK: 20.1 vs 19.0; LGA: 15.1 vs 15.0). This consistent pattern — mean > median — implies that each airport's arrival delay distribution is right-skewed, with a small number of severely delayed flights pulling the average upward relative to the typical flight.
>
> JFK shows the greatest spread with a standard deviation of ≈43.3 minutes, though it is essentially tied with EWR (also ≈43.3 min). LGA has a slightly smaller SD of ≈42.4 min, suggesting marginally less variability in arrival delays from that airport.

**Grading note (8 pts):** 2 pts for correctly identifying highest/lowest mean airport; 3 pts for the mean-vs-median comparison (must mention all three airports and correctly infer right skew); 3 pts for spread comparison using SD (must cite values). Deduct 1 pt if student confuses SD with IQR.